# 🎙️ TTS Studio — Coqui XTTS v2

### Instructions
1. `Runtime` → `Change runtime type` → **T4 GPU**
2. Exécuter la **cellule 1** puis `Runtime` → `Restart session`
3. Exécuter les **cellules 2 à 6** dans l'ordre
4. Copier l'URL ngrok et la coller dans l'app Next.js


In [ ]:
# ── Cellule 1 : Installation (puis Restart session) ──
import sys
print('Python :', sys.version)

# PyTorch compatible avec TTS 0.22.0
!pip install -q 'torch==2.3.1' 'torchaudio==2.3.1' --index-url https://download.pytorch.org/whl/cu121

# TTS + dépendances fixées
!pip install -q 'TTS==0.22.0' 'transformers==4.40.0' 'numpy==1.26.4' flask flask-cors pyngrok

print('✅ Installation terminée — faire Runtime > Restart session')

In [ ]:
# ── Cellule 2 : Vérification GPU ──
import torch
print('PyTorch :', torch.__version__)
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device :', torch.cuda.get_device_name(0))

In [ ]:
# ── Cellule 3 : Chargement du modèle XTTS v2 ──
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print(f'✅ Modèle chargé sur {device.upper()}')

In [ ]:
# ── Cellule 4 : Fonctions TTS ──
import re, os, shutil, subprocess, tempfile

NOMBRES = {
    0: 'zéro', 1: 'un', 2: 'deux', 3: 'trois', 4: 'quatre',
    5: 'cinq', 6: 'six', 7: 'sept', 8: 'huit', 9: 'neuf',
    10: 'dix', 11: 'onze', 12: 'douze', 13: 'treize', 14: 'quatorze',
    15: 'quinze', 16: 'seize', 17: 'dix-sept', 18: 'dix-huit', 19: 'dix-neuf',
    20: 'vingt', 30: 'trente', 40: 'quarante', 50: 'cinquante',
    60: 'soixante', 70: 'soixante-dix', 80: 'quatre-vingts', 90: 'quatre-vingt-dix',
    100: 'cent', 1000: 'mille',
}

def nombre_en_lettres(n):
    if n < 0: return 'moins ' + nombre_en_lettres(-n)
    if n in NOMBRES: return NOMBRES[n]
    if n < 70:
        d, u = (n // 10) * 10, n % 10
        if u == 0: return NOMBRES[d]
        if u == 1 and d in (20,30,40,50,60): return f'{NOMBRES[d]} et un'
        return f'{NOMBRES[d]}-{NOMBRES[u]}'
    if n < 80: return f'soixante-{nombre_en_lettres(n-60)}'
    if n < 100:
        r = n - 80
        return 'quatre-vingts' if r == 0 else f'quatre-vingt-{nombre_en_lettres(r)}'
    if n < 200:
        r = n - 100
        return 'cent' if r == 0 else f'cent {nombre_en_lettres(r)}'
    if n < 1000:
        c, r = n // 100, n % 100
        p = f"{nombre_en_lettres(c)} cent{'s' if r==0 else ''}"
        return p if r == 0 else f'{p} {nombre_en_lettres(r)}'
    if n < 1000000:
        m, r = n // 1000, n % 1000
        p = 'mille' if m == 1 else f'{nombre_en_lettres(m)} mille'
        return p if r == 0 else f'{p} {nombre_en_lettres(r)}'
    return str(n)

def normaliser_texte(texte):
    texte = re.sub(r'["\u201c\u201d\u00ab\u00bb]', '', texte)
    texte = re.sub(r'[—\u2014\u2013–]\s*', '', texte)
    texte = texte.replace('...', '.')
    texte = re.sub(r'(\d{1,2})h(\d{2})?', lambda m: f"{nombre_en_lettres(int(m.group(1)))} heures{' ' + nombre_en_lettres(int(m.group(2))) if m.group(2) else ''}", texte)
    texte = re.sub(r'(\d+)\s*%', lambda m: nombre_en_lettres(int(m.group(1))) + ' pour cent', texte)
    texte = re.sub(r'\b\d{1,6}\b', lambda m: nombre_en_lettres(int(m.group(0))), texte)
    texte = texte.replace(' : ', ', ').replace(' ; ', ', ')
    texte = texte.replace('\n', ' ')
    return re.sub(r'\s+', ' ', texte).strip()

def decouper_en_phrases(texte, max_len=220):
    segments = re.split(r'(?<=[.!?])\s+', texte)
    phrases = []
    for seg in segments:
        seg = seg.strip()
        if not seg: continue
        if len(seg) <= max_len:
            phrases.append(seg)
            continue
        buffer = ''
        for ss in seg.split(','):
            ss = ss.strip()
            if buffer and len(buffer) + len(ss) + 2 > max_len:
                phrases.append(buffer.strip())
                buffer = ss
            else:
                buffer = f'{buffer}, {ss}' if buffer else ss
        if buffer.strip(): phrases.append(buffer.strip())
    return [p for p in phrases if len(p) > 1]

def concatener_avec_silence(fichiers, output, silence_ms=300):
    if len(fichiers) == 1:
        shutil.copy2(fichiers[0], output)
        return
    silence_file = '/tmp/silence.wav'
    subprocess.run(['ffmpeg', '-y', '-f', 'lavfi', '-i',
        f'anullsrc=r=24000:cl=mono:d={silence_ms/1000}',
        '-t', str(silence_ms/1000), silence_file], capture_output=True)
    list_file = '/tmp/concat_list.txt'
    with open(list_file, 'w') as f:
        for i, fichier in enumerate(fichiers):
            f.write(f"file '{fichier}'\n")
            if i < len(fichiers) - 1:
                f.write(f"file '{silence_file}'\n")
    tmp_out = output + '.tmp.wav'
    subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', list_file,
        '-ar', '24000', '-ac', '1', '-c:a', 'pcm_s16le', tmp_out], capture_output=True)
    if output.endswith('.mp3'):
        subprocess.run(['ffmpeg', '-y', '-i', tmp_out, '-b:a', '192k', output], capture_output=True)
        os.remove(tmp_out)
    else:
        os.rename(tmp_out, output)

print('✅ Fonctions chargées')

In [ ]:
# ── Cellule 5 : Serveur Flask ──
import json, uuid, threading
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)
JOBS = {}

def run_tts(job_id, segments, ref_path, lang, speed, fmt, silence_ms):
    job = JOBS[job_id]
    job['status'] = 'running'
    job['total'] = len(segments)
    output_dir = f'/tmp/jobs/{job_id}'
    os.makedirs(output_dir, exist_ok=True)
    tmpdir = tempfile.mkdtemp()

    for idx, segment_brut in enumerate(segments):
        num = f'{idx+1:03d}'
        texte = normaliser_texte(segment_brut)
        phrases = decouper_en_phrases(texte)
        fichiers_phrases = []

        for i, phrase in enumerate(phrases):
            out_path = os.path.join(tmpdir, f'seg{num}_p{i:03d}.wav')
            try:
                tts.tts_to_file(
                    text=phrase,
                    speaker_wav=ref_path,
                    language=lang,
                    speed=speed,
                    file_path=out_path
                )
                if os.path.exists(out_path):
                    fichiers_phrases.append(out_path)
            except Exception as e:
                print(f'Erreur phrase {i}: {e}')

        if not fichiers_phrases:
            continue

        output_file = os.path.join(output_dir, f'segment_{num}.{fmt}')
        concatener_avec_silence(fichiers_phrases, output_file, silence_ms)
        for f in fichiers_phrases:
            if os.path.exists(f): os.remove(f)

        job['segments'].append(f'segment_{num}.{fmt}')
        job['done'] = idx + 1
        print(f'✅ segment_{num}.{fmt}')

    job['status'] = 'finished'
    try: os.rmdir(tmpdir)
    except: pass

@app.route('/generate', methods=['POST'])
def generate():
    json_file = request.files.get('json')
    ref_file  = request.files.get('reference')
    if not json_file or not ref_file:
        return jsonify({'error': 'Fichiers manquants'}), 400
    job_id     = str(uuid.uuid4())
    job_dir    = f'/tmp/jobs/{job_id}'
    os.makedirs(job_dir, exist_ok=True)
    ref_path   = os.path.join(job_dir, 'reference' + os.path.splitext(ref_file.filename)[1])
    ref_file.save(ref_path)
    segments   = json.load(json_file)
    lang       = request.form.get('lang', 'fr')
    speed      = float(request.form.get('speed', 1.0))
    fmt        = request.form.get('format', 'mp3')
    silence_ms = int(request.form.get('silence', 300))
    JOBS[job_id] = {'status': 'loading', 'segments': [], 'total': 0, 'done': 0}
    t = threading.Thread(target=run_tts, args=(job_id, segments, ref_path, lang, speed, fmt, silence_ms))
    t.daemon = True
    t.start()
    return jsonify({'jobId': job_id})

@app.route('/status/<job_id>')
def status(job_id):
    job = JOBS.get(job_id)
    if not job: return jsonify({'error': 'Job introuvable'}), 404
    return jsonify(job)

@app.route('/download/<job_id>/<segment>')
def download(job_id, segment):
    fpath = f'/tmp/jobs/{job_id}/{segment}'
    if not os.path.exists(fpath):
        return jsonify({'error': 'Fichier introuvable'}), 404
    return send_file(fpath, as_attachment=True)

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'engine': 'coqui-xtts-v2', 'device': device})

print('✅ Serveur Flask prêt')

In [ ]:
# ── Cellule 6 : Lancement ngrok + Flask ──
# 👉 Token gratuit sur https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'COLLE_TON_TOKEN_ICI'

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()

public_url = ngrok.connect(5000).public_url
print('━' * 50)
print(f'🌐 URL publique : {public_url}')
print('━' * 50)
print('👉 Colle cette URL dans le champ URL Colab de l\'app')

flask_thread = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
flask_thread.daemon = True
flask_thread.start()
print('✅ Serveur démarré !')